# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and visualizing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Identifier (@id): {metadata['@id']}")
print(f"License: {metadata['license']}")


## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s from the metadata.

In [ ]:
# Print available record sets and their @id
record_sets = dataset.record_sets()
print("Available record sets:")
for rs in record_sets:
    print(f"- Name: {rs.name} (@id: {rs.id})")

# For each record set, print fields and columns with their @id
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    print("Fields:")
    for field in rs.fields:
        print(f"  - Name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print("Columns:")
    for column in rs.columns:
        print(f"  - Name: {column.name}, @id: {column.id}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the `@id` values from the overview to reference entities.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Flatten record dictionaries if necessary
    dataframes[record_set_id] = pd.DataFrame(records)

# Show dataset columns for the first record set
if record_set_ids:
    ref_record_set_id = record_set_ids[0]
    print(f"Columns for record set (@id: {ref_record_set_id}):")
    print(dataframes[ref_record_set_id].columns.tolist())
    print(dataframes[ref_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. This example performs numeric filtering, normalization, and grouping using field `@id`s.

In [ ]:
# Choose record set and fields to explore
record_set_id = record_set_ids[0]  # just the first for illustration
df = dataframes[record_set_id]

# Select a numeric field for analysis
# You can find field @id from previous overview (e.g., 'cr:Age', 'cr:Interval_Between_Cancers')
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets():
    if rs.id == record_set_id:
        for field in rs.fields:
            if field.data_type in ('Integer', 'Float', 'Number') and numeric_field_id is None:
                numeric_field_id = field.id
            if field.data_type == 'Text' and group_field_id is None:
                group_field_id = field.id
        break

if numeric_field_id is None:
    print("No numeric field found for analysis.")
else:
    print(f"Using numeric field: {numeric_field_id}")
    threshold = 10
    # Filter records
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a text/categorical field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset (using field `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of the selected numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Example: Boxplot of numeric field grouped by group field
if group_field_id and numeric_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(y=df[numeric_field_id], x=df[group_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to use `mlcroissant` to load the FAIR^2 dataset, explore its metadata, extract tabular data using entity `@id` fields, process and visualize key attributes. These steps prepare the data for downstream clinical or machine learning analysis, ensuring reproducibility and transparency with Croissant schema.